In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

# ---------------------------------------------------------
# Bước 1: Đọc dữ liệu đã qua tiền xử lý
# ---------------------------------------------------------
train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')

# Xác định cột cần dự báo (Target)
target_col = 'Price' 

# Tách Features (X) và Target (y)
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test: X={X_test.shape}, y={y_test.shape}")

# ---------------------------------------------------------
# Bước 2: Huấn luyện mô hình Baseline (Linear Regression)
# ---------------------------------------------------------
print("\nMô hình Linear Regression")
model = LinearRegression()
model.fit(X_train, y_train)

# ---------------------------------------------------------
# Bước 3: Dự báo trên tập Test
# ---------------------------------------------------------
y_pred_log = model.predict(X_test)

# KHẮC PHỤC LỖI OVERFLOW (TRÀN SỐ)
# Cắt bỏ các giá trị dự báo quá vô lý (giới hạn log_price từ 0 đến 25)
# (25 tương đương e^25 = 72 tỷ tỷ, quá đủ cho giá nhà lớn nhất)
y_pred_log_clipped = np.clip(y_pred_log, a_min=0, a_max=25)

# ---------------------------------------------------------
# Bước 4: INVERSE TRANSFORM (Biến đổi ngược từ Log về Tiền Thật)
# ---------------------------------------------------------
y_train_real = np.expm1(y_train) 
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log_clipped) # Sử dụng biến đã cắt ngọn

# ---------------------------------------------------------
# Bước 5: Tính toán các Metrics dựa trên GIÁ TRỊ TIỀN THẬT
# ---------------------------------------------------------
mae = mean_absolute_error(y_test_real, y_pred_real)
mse = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse) 
r2 = r2_score(y_test_real, y_pred_real)
mape = mean_absolute_percentage_error(y_test_real, y_pred_real)

# Gói các kết quả vào một Dictionary
metrics_data = {
    "Chỉ số đánh giá (Metric)": ["MAE", "RMSE", "R² Score", "MAPE"],
    "Giá trị (Value)": [f"{mae:,.2f}", f"{rmse:,.2f}", f"{r2:.4f}", f"{mape * 100:,.2f}%"]
}

# Chuyển đổi thành dạng Bảng (DataFrame)
metrics_df = pd.DataFrame(metrics_data)

print("\n--- BẢNG KẾT QUẢ ĐÁNH GIÁ BASELINE ---")
display(metrics_df) # Dùng lệnh display thay vì print để bảng hiện ra đẹp nhất

# ---------------------------------------------------------
# Bước 6: Phân tích MAE theo phân khúc giá tiền thật
# ---------------------------------------------------------
results_df = pd.DataFrame({
    'Actual_Price': y_test_real,
    'Predicted_Price': y_pred_real
})

def categorize_price(price):
    if price < 5.0: return '1. Dưới 5 tỷ'
    elif price <= 15.0: return '2. Từ 5 - 15 tỷ'
    else: return '3. Trên 15 tỷ'

results_df['Segment'] = results_df['Actual_Price'].apply(categorize_price)

# Tạo danh sách trống để hứng dữ liệu
segment_results = []
grouped = results_df.groupby('Segment')

for segment, group in grouped:
    segment_mae = mean_absolute_error(group['Actual_Price'], group['Predicted_Price'])
    segment_results.append({
        "Phân khúc": segment,
        "Số lượng mẫu": f"{len(group):,}",
        "MAE": f"{segment_mae:,.2f}"
    })

# Chuyển thành DataFrame và hiển thị
segment_df = pd.DataFrame(segment_results)
print("\n--- BẢNG MAE THEO PHÂN KHÚC GIÁ ---")
display(segment_df)

Train: X=(40929, 55), y=(40929,)
Test: X=(10233, 55), y=(10233,)

Mô hình Linear Regression

--- BẢNG KẾT QUẢ ĐÁNH GIÁ BASELINE ---


,Chỉ số đánh giá (Metric),Giá trị (Value)
0,MAE,"14,176,688.85"
1,RMSE,"1,006,664,354.06"
2,R² Score,-23740.0649
3,MAPE,"750,467.58%"



--- BẢNG MAE THEO PHÂN KHÚC GIÁ ---


,Phân khúc,Số lượng mẫu,MAE
0,1. Dưới 5 tỷ,2,"8,322.94"
1,2. Từ 5 - 15 tỷ,1,"9,157.48"
2,3. Trên 15 tỷ,"10,230","14,180,843.71"


In [7]:
print(model.coef_)

[-3.40450025e-03  4.39106867e-02  2.00622197e-02  1.70105849e-02
  8.19055127e-02  3.98046027e-04  2.30686001e-02  2.48650707e-03
  2.51918939e-01 -1.00775639e+00 -3.44267498e-01 -1.60170078e+00
 -8.35923267e-01 -9.12594354e-01 -1.71504315e-01  4.52171223e-01
 -1.42302209e-01 -5.15871133e-01 -5.38140835e-01  5.21983437e-01
  9.17655654e-02 -2.07992991e-01 -9.11425885e-01 -5.51340511e-01
 -1.26245636e-01 -5.42077894e-01 -4.33219621e-01 -6.94639365e-01
 -5.83926949e-01 -6.34204834e-01 -5.40452016e-01 -5.02126397e-01
 -1.20155928e+00 -1.78494348e+00  1.13342377e+00  3.36597815e-01
  2.82804479e-01 -5.20156619e-01  1.18406225e+00  2.35600596e-01
  1.74217005e-02  1.00048340e-01  4.05859480e-01  1.46089829e-01
  1.08513505e-01  6.99041456e-02  1.29677890e-01  1.93206824e-01
  4.14351851e-02  2.75837202e-02 -4.87553082e-01 -2.60569539e-01
 -1.95680092e-01 -4.48965936e-01  1.33155228e-01]


In [4]:
train_df.describe()

,Area,Width,Length,Bedrooms,Bathrooms,Floors,Alley Width,Agent Listing Count,is_luxury,Price
count,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,4.092900e+04,40929.000000
mean,4.982427e-17,4.617859e-17,3.419994e-17,4.930346e-17,-7.556103e-17,3.055426e-17,-1.562434e-17,3.150908e-17,2.508574e-17,9.114637
std,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.486902
min,-7.425334e-03,-5.500898e-02,-5.202595e-02,-7.475987e-02,-1.262410e+00,-4.949458e-03,-3.597625e-01,-4.492501e-01,-9.047551e-01,1.064711
25%,-7.417501e-03,-3.535249e-02,-2.106968e-02,-1.629388e-02,-9.241815e-02,-4.948358e-03,-5.284458e-02,-4.396703e-01,-9.047551e-01,8.101981
50%,-7.412882e-03,-3.024181e-02,-2.106968e-02,-1.629388e-02,-9.241815e-02,-4.948358e-03,-5.284458e-02,-3.875139e-01,-9.047551e-01,8.853808
75%,-7.387577e-03,-1.962730e-02,-2.106968e-02,-1.629388e-02,-9.241815e-02,-4.948358e-03,-5.284458e-02,-7.516695e-03,1.105272e+00,9.903538
max,1.585404e+02,1.965099e+02,1.265873e+02,2.016914e+02,6.308712e+01,2.023066e+02,8.722695e+01,6.279999e+00,1.105272e+00,19.219207


In [2]:
import pandas as pd
train_df = pd.read_csv('../data/processed/train.csv')
print(train_df.columns.tolist())

['Area', 'Width', 'Length', 'Bedrooms', 'Bathrooms', 'Floors', 'Alley Width', 'Agent Listing Count', 'is_luxury', 'District_Huyện Bình', 'District_Huyện Cần', 'District_Huyện Củ', 'District_Huyện Hóc', 'District_Huyện Nhà', 'District_Huyện Thanh', 'District_Quận 1', 'District_Quận 10', 'District_Quận 11', 'District_Quận 12', 'District_Quận 2', 'District_Quận 3', 'District_Quận 4', 'District_Quận 5', 'District_Quận 6', 'District_Quận 7', 'District_Quận 8', 'District_Quận 9', 'District_Quận Bình', 'District_Quận Gò', 'District_Quận Phú', 'District_Quận Thủ', 'District_Quận Tân', 'District_TP. Hồ', 'District_Unknown', 'Property Type_Kho, nhà xưởng', 'Property Type_Khách sạn', 'Property Type_Nhà riêng', 'Property Type_Nhà trọ', 'Property Type_Văn phòng', 'Property Type_Đất', 'Position_Unknown', 'Position_Đường chính', 'Direction_Nam', 'Direction_Tây', 'Direction_Tây Bắc', 'Direction_Tây Nam', 'Direction_Unknown', 'Direction_Đông', 'Direction_Đông Bắc', 'Direction_Đông Nam', 'Road Type_Đườn